<a href="https://colab.research.google.com/github/codewithUswaFatima/urdu-ocr-codesaviours-si26-Uswa/blob/main/SI26_Uswa_Week4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1: Install Libraries

In [1]:
# Install compatible versions in ONE go (avoids version conflicts later)
!pip install -q -U transformers==4.46.3 tokenizers==0.20.3 sentencepiece protobuf accelerate datasets huggingface_hub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 15.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.35.1 which is incom

# Step 2: Import Libraries

In [2]:
import torch
import pandas as pd
from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel
)
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# Step 3: Load Dataset

In [3]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [4]:
CSV_FILE = "/content/gdrive/MyDrive/labels (2).csv"
IMAGE_FOLDER = "/content/data/processed"

In [5]:
import zipfile
import os

os.makedirs(IMAGE_FOLDER, exist_ok=True)

zip_files = [
    "/content/gdrive/MyDrive/synthetic.zip",
    "/content/gdrive/MyDrive/screenshots.zip",
]

for zip_path in zip_files:
    if not os.path.exists(zip_path):
        print(f"WARNING: {zip_path} not found. Check the exact path/filename in your Drive.")
        continue
    print(f"Extracting {zip_path} ...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(IMAGE_FOLDER)
    print(f"Done extracting {zip_path}")

print("\nAll extraction done. Contents of IMAGE_FOLDER:")
print(os.listdir(IMAGE_FOLDER))


Extracting /content/gdrive/MyDrive/synthetic.zip ...
Done extracting /content/gdrive/MyDrive/synthetic.zip
Extracting /content/gdrive/MyDrive/screenshots.zip ...
Done extracting /content/gdrive/MyDrive/screenshots.zip

All extraction done. Contents of IMAGE_FOLDER:
['synthetic', 'screenshots']


In [6]:
df = pd.read_csv(CSV_FILE)
df.head()

,image,text
0,raw/dataset/0.png,پشاور،بنوں (نمائندہ جنگ،اے ایف پی) بنوں میں اق...
1,raw/dataset/1.png,اسکے ساتھ ملحقہ علاقے کے عوام نے بجلی و گیس \n
2,raw/dataset/10.png,نے بجلی وگیس کی لوڈشیڈنگ کیخلاف مظاہرہ کیا۔ مظ...
3,raw/dataset/100.png,نہیں آ یا۔ آخریہ کونسا مذہب ہے؟کیا اسلام کے نا...
4,raw/dataset/1000.png,ضرور ہے۔ انہوں نے کہا کہ میرا احمدیوں سے کوئی \n


In [7]:
import os

# Build a set of all filenames that actually exist anywhere under IMAGE_FOLDER
existing_filenames = set()
for root, _, files in os.walk(IMAGE_FOLDER):
    for fname in files:
        existing_filenames.add(fname)

def image_exists(image_value):
    direct_path = os.path.join(IMAGE_FOLDER, image_value)
    if os.path.exists(direct_path):
        return True
    return os.path.basename(image_value) in existing_filenames

before_count = len(df)
df = df[df['image'].apply(image_exists)].reset_index(drop=True)
after_count = len(df)

print(f"Rows before cleanup: {before_count}")
print(f"Rows after cleanup:  {after_count}")
print(f"Dropped (missing images): {before_count - after_count}")


Rows before cleanup: 203
Rows after cleanup:  49
Dropped (missing images): 154


# Step 4: Load Processor

In [8]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from transformers import TrOCRProcessor,VisionEncoderDecoderModel

processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-printed"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

# Step 5: Create Dataset Class

In [9]:
import os

class UrduOCRDataset(Dataset):

  def __init__(self, dataframe, image_folder, processor):
    self.dataframe = dataframe
    self.image_folder = image_folder
    self.processor = processor

    # Build an index of basename -> full path by walking the image folder once.
    # This makes lookups work even if the CSV's folder structure doesn't exactly
    # match how the zip files were extracted (e.g. synthetic.zip vs screenshots.zip
    # having different internal folder layouts).
    self.filename_index = {}
    for root, _, files in os.walk(self.image_folder):
        for fname in files:
            self.filename_index[fname] = os.path.join(root, fname)

  def __len__(self):
    return len(self.dataframe)

  def _resolve_image_path(self, image_value):
    # 1) Try the exact path as given in the CSV
    direct_path = os.path.join(self.image_folder, image_value)
    if os.path.exists(direct_path):
        return direct_path

    # 2) Fall back to matching by filename only, anywhere under image_folder
    basename = os.path.basename(image_value)
    if basename in self.filename_index:
        return self.filename_index[basename]

    raise FileNotFoundError(
        f"Could not find image '{image_value}' (basename '{basename}') "
        f"anywhere under {self.image_folder}"
    )

  def __getitem__(self, idx):
    row = self.dataframe.iloc[idx]
    image_path = self._resolve_image_path(row['image'])
    image = Image.open(image_path).convert("RGB")
    pixel_values = self.processor(images=image, return_tensors="pt").pixel_values.squeeze()

    labels = self.processor.tokenizer(
        row["text"],
        padding="max_length",
        max_length=128,
        truncation=True,
    ).input_ids

    labels = [
        label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels
    ]

    return {
        "pixel_values": pixel_values,
        "labels": torch.tensor(labels)
    }


# Step 6: Train/Test Split

In [10]:
from sklearn.model_selection import train_test_split
train_df,test_df = train_test_split(
    df,
    test_size = 0.2,
    random_state= 42
)

# Step 7: Create Dataset Objects

In [11]:
train_dataset = UrduOCRDataset(train_df,IMAGE_FOLDER,processor)
test_dataset = UrduOCRDataset(test_df,IMAGE_FOLDER,processor)

# Step 8: Load TrOCR Model

In [12]:

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device}: device")
if device == "cpu":
  print("Warning No GPU Detected.")

Using cuda: device


In [13]:
processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-printed"
)
model = VisionEncoderDecoderModel.from_pretrained(
    "microsoft/trocr-base-printed"
)
model.to(device)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Config of the encoder: <class 'transformers.models.vit.modeling_vit.ViTModel'> is overwritten by shared encoder config: ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "qkv_bias": false,
  "transformers_version": "4.46.3"
}

Config of the decoder: <class 'transformers.models.trocr.modeling_trocr.TrOCRForCausalLM'> is overwritten by shared decoder config: TrOCRConfig {
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_cross_attention": true,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classifier_dropout": 0.0,
  "cross_attention_hidden_size": 768,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder

generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

VisionEncoderDecoderModel(
  (encoder): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTSdpaAttention(
            (attention): ViTSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=False)
              (key): Linear(in_features=768, out_features=768, bias=False)
              (value): Linear(in_features=768, out_features=768, bias=False)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linea

# Step 9: Configure Model

In [14]:
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
print("Model loaded successfully!")


Model loaded successfully!


In [15]:
from torch.utils.data import DataLoader
from torch.optim import AdamW

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False
)

optimizer = AdamW(
    model.parameters(),
    lr=5e-5
)

print("Training batches:", len(train_loader))
print("Ready to train!")

Training batches: 10
Ready to train!


# Step 10: Training Loop


In [16]:
num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    print("-" * 30)

    for batch_idx, batch in enumerate(train_loader):
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 10 == 0:
            print(f"  Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1} complete | Average Loss: {avg_loss:.4f}")

print("\nTraining complete!")



Epoch 1/3
------------------------------
  Batch 0/10 | Loss: 17.4967
Epoch 1 complete | Average Loss: 8.1809

Epoch 2/3
------------------------------
  Batch 0/10 | Loss: 4.6903
Epoch 2 complete | Average Loss: 4.4439

Epoch 3/3
------------------------------
  Batch 0/10 | Loss: 3.8722
Epoch 3 complete | Average Loss: 3.8344

Training complete!


# Step 11: Evaluate the Model


Now we test the model on images it has never seen before (the test set). The model is switched to `eval()` mode so it does not update its weights during this step.


In [18]:
model.eval()

print("=== Model Evaluation on Test Images ===")
print()

correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"]

        generated_ids = model.generate(pixel_values, max_new_tokens=128)
        generated_text = processor.batch_decode(
            generated_ids, skip_special_tokens=True
        )

        # Replace -100 (used to mask padding during loss calc) back to pad_token_id
        # before decoding, otherwise the tokenizer errors out on the invalid id.
        decodable_labels = labels.clone()
        decodable_labels[decodable_labels == -100] = processor.tokenizer.pad_token_id

        actual_text = processor.batch_decode(
            decodable_labels, skip_special_tokens=True
        )

        for pred, actual in zip(generated_text, actual_text):
            total += 1
            if pred.strip() == actual.strip():
                correct += 1
            print(f"Predicted: {pred}")
            print(f"Actual:    {actual}")
            print()

accuracy = (correct / total) * 100 if total > 0 else 0
print(f"Accuracy: {accuracy:.1f}% ({correct}/{total} correct)")

=== Model Evaluation on Test Images ===

Predicted: �������������������������������������������������������������������������������������������������������������������������������
Actual:    پاکستان کے صوبہ خیبرپختونخوا میں حالیہ تیز بارشوں، آندھی اور فلیش فلڈ کے نتیجے میں مختلف حادثات م�

Predicted: �������������������������������������������������������������������������������������������������������������������������������
Actual:    سچائی ہمیشہ فتح یاب ہوتی ہے

Predicted: �������������������������������������������������������������������������������������������������������������������������������
Actual:    درخت لگانا نیکی کا کام ہے

Predicted: �������������������������������������������������������������������������������������������������������������������������������
Actual:    آج کا موسم خوشگوار ہے

Predicted: �������������������������������������������������������������������������������������������������������������������������������
Actual:    قطری وزارتِ خارجہ کے مطابق پاک

# Step 12: Save the Model to Google Drive


In [19]:
save_path = "/content/gdrive/MyDrive/SI26-urdu-ocr-model"

model.save_pretrained(save_path)
processor.save_pretrained(save_path)

print(f"Model saved to Google Drive: {save_path}")
print("You can load this model again next week without retraining")


Model saved to Google Drive: /content/gdrive/MyDrive/SI26-urdu-ocr-model
You can load this model again next week without retraining
